# Lesson 2.1 — 环境与 rollout 循环

本 notebook 建立 GPT/agent-loop 类比所依赖的直觉：Gymnasium 的调用模式**就是**具身智能体循环（embodied agent loop）。

```text
o_t  ->  policy  ->  a_t  ->  env.step  ->  o_{t+1}, r, terminated, truncated
```

改编自已归档的探针脚本
`archive/lesson_0_1/smoke_test_maniskill.py` 与
`archive/lesson_0_1/collect_pickcube_random.py`；它们在 Lesson 0–1 中确立了这一模式，现已冻结。


## 2.1.1 — 创建与 reset


In [1]:
import gymnasium as gym
import mani_skill.envs
import numpy as np
import torch

torch.set_printoptions(precision=4, sci_mode=False)


def make_env(obs_mode="state", control_mode="pd_joint_delta_pos", seed=0):
    env = gym.make(
        "PickCube-v1",
        obs_mode=obs_mode,
        control_mode=control_mode,
        num_envs=1,
    )
    env.reset(seed=seed)
    return env


env = gym.make("PickCube-v1", obs_mode="state", control_mode="pd_joint_delta_pos", num_envs=1)

print("environment :", type(env.unwrapped).__name__)
print("obs space   :", env.observation_space)
print("action space:", env.action_space)
print("control freq:", env.unwrapped.control_freq, "Hz")

obs, info = env.reset(seed=0)
print("\nreset -> obs shape:", obs.shape, "dtype:", obs.dtype)
print("info keys:", list(info.keys()))

2026-09-22 11:14:22,321 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1113: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 304: OS call failed or operation not supported on this OS (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  r = torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count


environment : PickCubeEnv
obs space   : Box(-inf, inf, (1, 42), float32)
action space: Box(-1.0, 1.0, (8,), float32)
control freq: 20 Hz

reset -> obs shape: torch.Size([1, 42]) dtype: torch.float32
info keys: ['elapsed_steps', 'success', 'is_obj_placed', 'is_robot_static', 'is_grasped', 'reconfigure']


## 2.1.2 — 单步：五个返回值

`env.step(action)` 返回 `(observation, reward, terminated, truncated, info)`。
两个彼此独立的停止信号是常见的 bug 来源：

- `terminated` —— 任务以成功或不可恢复的失败告终；
- `truncated` —— episode 触及 step 上限或时间上限。

一个被 `truncated` 的 episode 并不能说明成功与否。本项目的 source trajectory
正是以这种方式结束，因此 `notes/progress.md` 中记录为 `success_any=False`。


In [2]:
action = env.action_space.sample()
print("action:", action, "shape:", action.shape)

next_obs, reward, terminated, truncated, info = env.step(action)

print("\nnext_obs shape:", next_obs.shape)
print("reward        :", reward)
print("terminated    :", terminated)
print("truncated     :", truncated)
print("info          :", info)

action: [ 0.62323415 -0.5443854   0.82626545  0.22426002 -0.08090118 -0.9197369
 -0.27873144  0.39224908] shape: (8,)

next_obs shape: torch.Size([1, 42])
reward        : tensor([0.0617])
terminated    : tensor([False])
truncated     : tensor([False])
info          : {'elapsed_steps': tensor([1], dtype=torch.int32), 'success': tensor([False]), 'is_obj_placed': tensor([False]), 'is_robot_static': tensor([False]), 'is_grasped': tensor([False])}


## 2.1.3 — 一个完整的随机 episode

一次 rollout 就是这个循环本身。这里刻意使用随机 action：目标是验证机制，而不是解决任务。

注意 T 与 state 的记账方式：循环执行 T 步并存储 **T** 个 observation、
action 和 reward。最后一个 action *之后* 的 observation 不会被存储，除非你显式保留它。
这个 off-by-one 问题正是 `2.3_time_alignment.ipynb` 的主题。


In [3]:
def collect_random_episode(env, seed=0, max_steps=50, verbose=False):
    """Run a random rollout and record the transitions.

    Returns T observations, T actions, T rewards where T = max_steps.
    """
    obs, info = env.reset(seed=seed)
    observations, actions, rewards = [], [], []
    terminated_flags, truncated_flags = [], []

    for step in range(max_steps):
        action = env.action_space.sample()
        next_obs, reward, terminated, truncated, info = env.step(action)

        observations.append(obs[0].cpu().numpy())
        actions.append(action.copy())
        rewards.append(float(reward.item()))
        terminated_flags.append(bool(terminated.item()))
        truncated_flags.append(bool(truncated.item()))

        if verbose and step < 5:
            print(f"  step {step:2d}: reward={float(reward.item()):+.4f} "
                  f"terminated={bool(terminated.item())} truncated={bool(truncated.item())}")

        obs = next_obs
        if bool(terminated.item()) or bool(truncated.item()):
            break

    return {
        "observations": np.asarray(observations, dtype=np.float32),
        "actions": np.asarray(actions, dtype=np.float32),
        "rewards": np.asarray(rewards, dtype=np.float32),
        "terminated": np.asarray(terminated_flags),
        "truncated": np.asarray(truncated_flags),
        "info": info,
    }


episode = collect_random_episode(env, seed=0, max_steps=50, verbose=True)

print("\nobservations:", episode["observations"].shape)
print("actions     :", episode["actions"].shape)
print("rewards     :", episode["rewards"].shape)
print("total reward:", episode["rewards"].sum())
print("any success :", "success" in episode["info"])
print("terminated  :", episode["terminated"].any(), "| truncated:", episode["truncated"].any())

  step  0: reward=+0.0560 terminated=False truncated=False
  step  1: reward=+0.0495 terminated=False truncated=False
  step  2: reward=+0.0522 terminated=False truncated=False
  step  3: reward=+0.0511 terminated=False truncated=False
  step  4: reward=+0.0429 terminated=False truncated=False

observations: (50, 42)
actions     : (50, 8)
rewards     : (50,)
total reward: 2.1162405
any success : True
terminated  : False | truncated: True


## 2.1.4 — 这里的 “success” 究竟指什么

`PickCube-v1` 并不把 “gripper 正抓着 cube” 视为 success。它的评测要求 cube
被 **放置在 goal threshold 之内**，*并且* 机器人处于 **static** 状态。仅完成抓取不算 success。

这对 Lesson 2.9 及之后的内容很重要：不能因为机械臂的运动看起来合理，就把一段
demonstration 当作 expert data。必须存在显式的 success 信号。


In [4]:
env.reset(seed=0)
print("goal threshold   :", env.unwrapped.goal_thresh)
print("cube half size   :", env.unwrapped.cube_half_size)
print("cube position    :", env.unwrapped.cube.pose.p[0])
print("goal position    :", env.unwrapped.goal_site.pose.p[0])

distance = (env.unwrapped.cube.pose.p[0] - env.unwrapped.goal_site.pose.p[0]).norm()
print(f"current cube-to-goal distance: {distance.item():.4f} m")
print("episode starts far from the goal, and random actions do not close that gap.")

goal threshold   : 0.025
cube half size   : 0.02
cube position    : tensor([-0.0007,  0.0536,  0.0200])
goal position    : tensor([ 0.0268, -0.0020,  0.2889])
current cube-to-goal distance: 0.2760 m
episode starts far from the goal, and random actions do not close that gap.


## 小结

1. `reset` 返回 `(obs, info)`；`step` 返回五个值，其中 `terminated` 与
   `truncated` 是 **相互独立** 的停止条件。
2. rollout 循环就是具身智能体循环。policy 是唯一可替换的部分。
3. T 步产生 T 个 observation、T 个 action、T 个 reward。
4. 在 `PickCube-v1` 中，success 要求 cube 被放置在 goal threshold 之内，并且机器人
   处于 static 状态 —— 仅完成抓取是不够的。随机 rollout 只是 pipeline 的
   fixture，绝不是 expert demonstration。

下一步：dataset 层，从 `2.3_time_alignment.ipynb` 开始。
